# Getting Goober to walk

Use a CPU runtime in Colab and run the cells from the top. Pick both ZIPs in the same upload window. One has the goose, the other has the walking policy you want to start from. Model 4 works here and uses the same 36 inputs.

I wanted to see what happens when Goober gets a little less grip. The floor changes between training runs, and there’s a slippery-floor test at the end.

In [ ]:
%pip install -q "mujoco==3.3.7" "gymnasium==1.2.2" "stable-baselines3==2.7.1"

In [ ]:
import os
import sys
from pathlib import Path
from zipfile import ZipFile
from io import BytesIO
from google.colab import files

for name in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ[name] = "1"

print("Select the Goober robot ZIP and the PPO checkpoint ZIP together.")
uploads = files.upload()
robot_path = checkpoint = None
for name, content in uploads.items():
    with ZipFile(BytesIO(content)) as archive:
        entries = archive.namelist()
    if "goober.xml" in entries:
        robot_path = Path(name).resolve()
    elif "policy.pth" in entries:
        checkpoint = Path(name).resolve()
if robot_path is None or checkpoint is None:
    raise ValueError("Upload both Goober-MuJoCo-legs.zip and a Goober PPO checkpoint.")
print("Model:", robot_path.name, "| Starting policy:", checkpoint.name)

## Give each CPU worker its own goose

In [ ]:
%%writefile goober_cpu_env.py
from zipfile import ZipFile

import gymnasium as gym
from gymnasium import spaces
import mujoco
import numpy as np
from stable_baselines3.common.monitor import Monitor

class GooberWalkEnv(gym.Env):
    def __init__(self, model, grip_range=None):
        super().__init__()
        self.model = model
        self.data = mujoco.MjData(model)
        self.torso = model.body("torso").id
        self.floor = model.geom("floor").id
        self.feet = [model.body(s + "_foot").id for s in ("left", "right")]
        self.grip_range = grip_range
        self.nominal_friction = model.geom_friction[:, 0].copy()
        self.contact_jacobian = np.zeros((3, model.nv))
        joint_ids = model.actuator_trnid[:, 0]
        self.qadr = model.jnt_qposadr[joint_ids]
        self.vadr = model.jnt_dofadr[joint_ids]
        self.low, self.high = model.actuator_ctrlrange.T
        self.dt = 10 * model.opt.timestep
        self.target = 0.12
        self.max_steps = round(8.0 / self.dt)
        self.last = np.zeros(model.nu, dtype=np.float32)
        self.action_space = spaces.Box(-1.0, 1.0, shape=(model.nu,), dtype=np.float32)
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(36,), dtype=np.float32)

    def _up(self):
        return self.data.xmat[self.torso].reshape(3, 3)[2, :]

    def _floor_contacts(self):
        touch = np.zeros(2)
        slip_speed = 0.0
        body_contact = False
        for c in self.data.contact:
            if c.dist > 0 or self.floor not in (c.geom1, c.geom2):
                continue
            other = c.geom2 if c.geom1 == self.floor else c.geom1
            body = self.model.geom_bodyid[other]
            if body in self.feet:
                touch[self.feet.index(body)] = 1
                # Horizontal speed of the actual contact point on the rigid foot.
                mujoco.mj_jac(self.model, self.data, self.contact_jacobian, None, c.pos, body)
                velocity = self.contact_jacobian[:2] @ self.data.qvel
                slip_speed = max(slip_speed, float(np.linalg.norm(velocity)))
            else:
                body_contact = True
        return touch, slip_speed, body_contact

    def _count_landing(self, touch, body_contact):
        x = self.data.xpos[self.feet, 0]
        counted = 0
        for i in range(2):
            supported = bool(touch[1-i]) and not body_contact
            if not touch[i]:
                if self.air_ticks[i] == 0:
                    self.swing_valid[i] = self.had_stance[i]
                self.swing_valid[i] &= supported
                self.air_ticks[i] += 1
            else:
                if (self.swing_valid[i] and self.air_ticks[i] >= 3 and not body_contact
                        and x[i] - self.last_stance_x[i] >= 0.01 and self.last_step != i):
                    counted = 1
                    self.last_step = i
                self.air_ticks[i] = 0
                self.had_stance[i] = True
                self.last_stance_x[i] = x[i]
        return counted

    def _obs(self, touch):
        d = self.data
        return np.concatenate((self._up(), [d.xpos[self.torso, 2]],
                               d.qvel[:6], d.qpos[self.qadr], d.qvel[self.vadr],
                               self.last, touch)).astype(np.float32)

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        mujoco.mj_resetData(self.model, self.data)
        self.data.qpos[self.qadr] += self.np_random.uniform(-0.015, 0.015, self.model.nu)
        self.last.fill(0)
        self.steps = 0
        mujoco.mj_forward(self.model, self.data)
        if self.grip_range is not None:
            factor = self.np_random.uniform(*self.grip_range)
            self.model.geom_friction[:, 0] = self.nominal_friction * factor
            mujoco.mj_forward(self.model, self.data)
        touch, _, _ = self._floor_contacts()
        self.had_stance = touch.astype(bool)
        self.air_ticks = np.zeros(2, dtype=int)
        self.swing_valid = np.zeros(2, dtype=bool)
        self.last_stance_x = self.data.xpos[self.feet, 0].copy()
        self.last_step = -1
        self.body_ticks = 0
        return self._obs(touch), {}

    def step(self, action):
        d = self.data
        action = np.clip(action, -1, 1)
        d.ctrl[:] = (self.low + self.high + action * (self.high - self.low)) / 2
        xy = d.xpos[self.torso, :2].copy()
        mujoco.mj_step(self.model, d, nstep=10)
        mujoco.mj_forward(self.model, d)
        vx, vy = (d.xpos[self.torso, :2] - xy) / self.dt
        up = float(self._up()[1])
        fallen = up < 0.65 or d.xpos[self.torso, 2] < 0.065
        tracking = 1.0 - min(abs(vx - self.target) / self.target, 2.0)
        reward = (tracking + 0.25 * up - 0.4 * abs(vy)
                  - 0.06 * np.mean((action - self.last)**2) - 5.0 * fallen)
        touch, slip_speed, body_contact = self._floor_contacts()
        self.body_ticks = self.body_ticks + 1 if body_contact else 0
        dragging = self.body_ticks >= 3
        landing = self._count_landing(touch, body_contact)
        reward += 3.0 * landing - 2.0 * slip_speed - 1.0 * dragging
        self.last[:] = action
        self.steps += 1
        return (self._obs(touch), float(reward), bool(fallen), self.steps >= self.max_steps,
                {"speed": float(vx), "fallen": bool(fallen), "landing": landing,
                 "slip_speed": slip_speed, "dragging": dragging})

def load_robot(path):
    with ZipFile(path) as archive:
        meshes = {n: archive.read(n) for n in archive.namelist() if n.endswith(".stl")}
        model = mujoco.MjModel.from_xml_string(archive.read("goober.xml").decode(), meshes)
    if model.nu != 8 or model.nq != 15:
        raise ValueError("Expected the articulated Goober model with eight actuators.")
    return model

def make_worker(path):
    return Monitor(GooberWalkEnv(load_robot(path), grip_range=(0.8, 1.2)))

## Is it quicker to run more than one?

Colab tries one, two or four Goobers at once, depending on how many cores are available. They all start from the same saved policy. Each setup collects 1,024 steps per update in total. The timer starts after warm-up and includes the time spent learning. The fastest setup gets used below. The practice policies get thrown away.

In [ ]:
from functools import partial
from time import perf_counter
import numpy as np
import torch
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv
from goober_cpu_env import GooberWalkEnv, load_robot, make_worker

torch.set_num_threads(1)
cores = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else (os.cpu_count() or 1)
quota_path = Path("/sys/fs/cgroup/cpu.max")
if quota_path.is_file():
    quota, period = quota_path.read_text().split()
    if quota != "max":
        cores = min(cores, max(1, int(quota) // int(period)))
print("Available CPU slots:", cores)

def environments(count):
    factories = [partial(make_worker, robot_path) for _ in range(count)]
    if count == 1:
        return DummyVecEnv(factories)
    return SubprocVecEnv(factories, start_method="spawn")

def load_policy(path, env):
    # Use this simulation's spaces and fresh rollout state across Colab versions.
    fresh = dict(observation_space=env.observation_space, action_space=env.action_space,
                 _last_obs=None, _last_episode_starts=None, ep_info_buffer=None, ep_success_buffer=None)
    return PPO.load(path, env=env, device="cpu", n_steps=1024 // env.num_envs,
                    batch_size=64, verbose=0, custom_objects=fresh)

speeds = {}
for count in (1, 2, 4):
    if count > cores:
        continue
    envs = environments(count)
    try:
        trial = load_policy(checkpoint, envs)
        trial.learn(1024, reset_num_timesteps=False)
        start = perf_counter()
        trial.learn(3072, reset_num_timesteps=False)
        speeds[count] = 3072 / (perf_counter() - start)
        print(f"{count} Goober(s): {speeds[count]:.0f} total training steps/s", flush=True)
    finally:
        envs.close()
workers = max(speeds, key=speeds.get)
print(f"Selected {workers} Goober(s) for training.")

## What counts as a step?

Goober can cover ground by sliding. For a step to count, one foot has to stay up for at least 60 ms, move forward at least 1 cm and land while the other foot supports him. Then it’s the other foot’s turn.

Every policy gets the same ten starts on the original floor. Alongside falls and steps, you’ll see slip in meters and body drag in seconds. Slip adds up the movement of the fastest sliding foot contact. Drag starts counting on the third reading in a row where something besides a foot touches the floor.

In [ ]:
import shutil
from tempfile import mkdtemp

robot = load_robot(robot_path)
validation_seeds = range(3100, 3110)

def evaluate(policy, seeds, model=None):
    env = GooberWalkEnv(robot if model is None else model)
    rows = []
    for seed in seeds:
        obs, _ = env.reset(seed=seed)
        x0 = float(env.data.xpos[env.torso, 0])
        steps = slip_m = drag_s = 0
        for _ in range(env.max_steps):
            action, _ = policy.predict(obs, deterministic=True)
            obs, _, fell, timed_out, info = env.step(action)
            steps += info["landing"]
            slip_m += info["slip_speed"] * env.dt
            drag_s += info["dragging"] * env.dt
            if fell or timed_out:
                break
        forward = float(env.data.xpos[env.torso, 0]) - x0
        rows.append([fell, forward, steps, slip_m, drag_s])
    env.close()
    falls, forward, steps, slip, drag = np.mean(rows, axis=0)
    gait_score = forward + 0.02 * steps - slip - 0.10 * drag
    rank = (-sum(int(row[0]) for row in rows), gait_score)
    summary = (f"{-rank[0]}/{len(rows)} falls, {forward:.2f} m forward, "
               f"{steps:.1f} steps, {slip:.2f} m slip, {drag:.2f} s body drag per trial")
    return rank, summary

run_dir = Path(mkdtemp(prefix="goober_run_", dir=".")).resolve()
best_path = run_dir / "goober_gait_best_ppo.zip"
shutil.copyfile(checkpoint, best_path)
eval_env = DummyVecEnv([lambda: GooberWalkEnv(robot)])
try:
    starting_policy = load_policy(best_path, eval_env)
    best_rank, best_summary = evaluate(starting_policy, validation_seeds)
    best_steps = starting_policy.num_timesteps
finally:
    eval_env.close()
attempt = 0
print(f"Starting best at {best_steps:,} steps: {best_summary}.")

## Try five more times

Each try starts with the best policy saved so far and trains for 10,240 steps. Floor grip varies from 80% to 120% of the original value. After training, Goober goes back on the original floor for the same ten trials.

The policy with fewer falls wins. If they tie, forward distance and alternating steps help the score, and slipping or dragging brings it down. Run this cell again if you want another five tries.

In [ ]:
envs = environments(workers)
try:
    for _ in range(5):
        attempt += 1
        policy = load_policy(best_path, envs)
        policy.set_random_seed(10_000 + attempt)
        policy.learn(10_240, reset_num_timesteps=False)
        rank, summary = evaluate(policy, validation_seeds)
        improved = rank > best_rank
        if improved:
            temporary = run_dir / "candidate.zip"
            policy.save(temporary)
            temporary.replace(best_path)
            best_rank, best_summary, best_steps = rank, summary, policy.num_timesteps
        status = "New best" if improved else "Kept previous best"
        print(f"Attempt {attempt}: {summary}. {status}.", flush=True)
finally:
    envs.close()
print(f"Best checkpoint: {best_steps:,} steps, {best_summary}.")

## Does it still work with less grip?

Here Goober gets ten fresh starts. First he tries them on the original floor, then he tries those same starts with the grip turned down to 70%. Check how many times he fell and how far he got on each floor. The ZIP that downloads is the policy you can start with next time.

In [ ]:
test_env = DummyVecEnv([lambda: GooberWalkEnv(robot)])
try:
    winner = load_policy(best_path, test_env)
    test_robot = load_robot(robot_path)
    for grip in (1.0, 0.7):
        test_robot.geom_friction[:, 0] = robot.geom_friction[:, 0] * grip
        _, summary = evaluate(winner, range(4100, 4110), test_robot)
        print(f"Test at {grip:.0%} of original grip: {summary}.")
finally:
    test_env.close()
files.download(str(best_path))